# T7 It6: after the blind check (diagnosis only; nothing here counts as evidence on NUDT 606)

The blind set had two wrong answers under the frozen PC rule: photo 182 (8 inliers, P 0.96; the baseline answers it too) and photo 2071
(**3 inliers**, DINO z 2.64, P 0.95, found in the wide queries; a new wrong answer). The judge can rate a 3-inlier candidate highly when DINO z is
large. Question: what would a minimum inlier count for an *answer* cost on the seen sets (leave one set out, same frozen rule otherwise)? If it is
cheap, it is the candidate rule for the next blind set. It is **not** re-scored on 606 here, since 606 has now been seen.

In [ ]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import pandas as pd
from train.checks.candidate_judge import FEATURES_CTX, FROZEN, add_context, add_features, baseline, held_out_probabilities, leave_one_set_out, paired_gain
RES = ROOT / "tests/Test_Iteration_7/results"
SETS = ["official", "nudt101", "nudt202", "nudt404", "nudt505"]
table = add_context(add_features(pd.concat([pd.read_csv(RES / f"signals/{s}.csv.gz") for s in SETS], ignore_index=True))).reset_index(drop=True)
PROBS = held_out_probabilities(table, SETS, FEATURES_CTX)
blind = pd.read_csv(RES / "it5_blind606_photos.csv"); display(blind[blind.pts < 0])

In [ ]:
bases = {"pc": baseline(table, 8, 11, min_choice=3), "jetson_early": baseline(table, 8, 8, jetson=True, m=5)}
out = []
for mc in (3, 4, 5):
    for variant, kw in {"pc": {}, "jetson_early": dict(jetson=True, m=5, early_tau=0.98)}.items():
        s, rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, cut_rule="ltt", alpha=FROZEN["alpha"], tau_floor=FROZEN["tau_floor"],
                                    wide_tau=FROZEN["wide_tau"], override=FROZEN["override"], min_choice=mc, **kw)
        g = paired_gain(bases[variant], rows, SETS)
        s["cell"] = s.score.map("{:.2f}".format) + " / " + s.wrong.astype(str) + "w / " + s.held.map(g)
        out.append(s.assign(rule=f"{variant}, min inliers {mc}"))
mi = pd.concat(out)
display(mi.pivot(index="rule", columns="held", values="cell")[SETS])
mi.to_csv(RES / "it6_min_inliers_loso.csv", index=False)

## Reading

A minimum of 4 inliers for an answer costs ~0.2 on official and nothing elsewhere, with 0 wrong on every held-out set; 5 costs more on the Jetson.
**Candidate for the next blind set: min inliers 4** (the 2071 kind of answer is then impossible). Photo 182 (8 inliers, P 0.96) is a look-alike
that every rule tested answers, the baseline included; no post-matching signal here separates it.